# Excuse Judge: a Zoro-1 notebook demo

<!-- Rename this notebook and its folder, replace the title and this paragraph. -->

Give Zoro-1 an excuse for late homework. Its **decision head** rates how believable the excuse is (a `score` over 5
levels) and whether it's funny (a `noul`, yes/no), both with calibrated probabilities. Then the **chat** endpoint
answers as the teacher.

**Run it:** start a Zoro-1 server ([Run Zoro-1](../README.md#run-zoro-1)) and run all cells. No server? Set
`MOCK = True` in the next cell (or `ZORO_MOCK=1` in the environment) and it runs with fake answers.

*Outputs are cleared in this notebook: run it to see results.*

In [ ]:
# Setup: the only dependency is httpx.
# %pip install -q httpx
import os

ZORO_URL = os.environ.get("ZORO_URL", "http://127.0.0.1:8080")   # MLX server; SGLang: http://GPU_HOST:30001
MOCK = os.environ.get("ZORO_MOCK", "0") == "1"                   # True: no model needed

## A tiny Zoro-1 client

Two calls cover most demos: `decide` (`POST /v1/systemone`) and `chat` (`POST /v1/chat/completions`, thinking off).
The mock returns deterministic fake answers in the same shape, so the notebook runs anywhere.

In [ ]:
import hashlib, json, random
import httpx

def _mock_decide(state, questions):
    rng = random.Random(hashlib.sha256(json.dumps([state, questions], sort_keys=True).encode()).digest())
    out = {}
    for qid, q in questions.items():
        if q["type"] == "noul":
            out[qid] = {"type": "noul", "noul": rng.random()}
            continue
        keys = list(q["criteria"]) if q["type"] == "choice" else [str(i) for i in range(len(q["criteria"]))]
        w = [rng.random() for _ in keys]
        probs = {k: x / sum(w) for k, x in zip(keys, w)}
        if q["type"] == "choice":
            out[qid] = {"type": "choice", "choice": max(probs, key=probs.get), "probabilities": probs}
        else:
            out[qid] = {"type": "score", "score": sum(int(k) * p for k, p in probs.items()), "probabilities": probs}
    return out

def decide(state, questions, image=None):
    """{question id: answer}. Types: noul (yes/no), choice (pick one), score (ordered levels)."""
    if MOCK:
        return _mock_decide(state, questions)
    body = {"model": "zoro-1", "state": state, "questions": questions, "reasoning": False}
    if image:
        body["images"] = [image]   # one data URL per request
    r = httpx.post(f"{ZORO_URL}/v1/systemone", json=body, timeout=120)
    r.raise_for_status()
    return r.json()["answers"]

def chat(messages, max_tokens=200):
    if MOCK:
        return "(mock reply) " + messages[-1]["content"][:60]
    r = httpx.post(f"{ZORO_URL}/v1/chat/completions", timeout=120, json={
        "model": "zoro-1", "messages": messages, "max_tokens": max_tokens, "temperature": 0.7,
        "chat_template_kwargs": {"enable_thinking": False}})
    r.raise_for_status()
    return (r.json()["choices"][0]["message"].get("content") or "").strip()

if not MOCK:   # fail loudly if the server isn't up
    print("Zoro-1 server:", httpx.get(f"{ZORO_URL}/v1/models", timeout=5).json()["data"][0]["id"])
else:
    print("Mock mode: answers are fake.")

## Rate some excuses

One request, two questions about the same state. They share one prefill, so the second question is nearly free.
`score` returns E[level] over the levels; dividing by the last index turns it into 0–1.

In [ ]:
LEVELS = ["obviously fake", "hard to believe", "could be true", "believable", "definitely true"]

def judge(excuse):
    a = decide({"situation": "A student is late with their homework and gives an excuse.", "excuse": excuse},
               {"believable": {"type": "score", "instructions": "How believable is this excuse?", "criteria": LEVELS},
                "funny": {"type": "noul", "instructions": "Is the excuse funny?"}})
    return a["believable"]["score"] / (len(LEVELS) - 1), a["funny"]["noul"]

excuses = ["my dog ate it", "the bus was 40 minutes late", "a time traveler took it to 3024 for safekeeping"]
results = {e: judge(e) for e in excuses}
for e, (b, f) in results.items():
    print(f"{b:>5.0%} believable  {f:>5.0%} funny   {e}")

## The teacher answers

The chat endpoint, with thinking off for a quick reply.

In [ ]:
worst = min(results, key=lambda e: results[e][0])
reply = chat([{"role": "system", "content": "You are a strict but fair teacher. Answer in one sentence."},
              {"role": "user", "content": f"My excuse: {worst}"}])
print(f"Excuse: {worst}\nTeacher: {reply}")

## Try your own

Change the excuse and run the cell.

In [ ]:
b, f = judge("my code was still compiling")
print(f"{b:.0%} believable, {f:.0%} funny")

## Checks

Keep a last cell of small assertions: CI runs the notebook top to bottom in mock mode, and these catch a broken
demo.

In [ ]:
assert all(0 <= b <= 1 and 0 <= f <= 1 for b, f in results.values())
assert isinstance(reply, str) and reply
print("ok")